# U03 · Evaluar bien — métricas y validación en clínica

> **Para quien nunca ha programado.** El código ya está escrito y comentado. Tu trabajo es **entender qué hace y por qué**, con criterio clínico. Ejecuta las celdas de arriba abajo (▶ o **Mayús + Enter**); para lanzarlo todo de golpe: *Entorno de ejecución → Ejecutar todo*.

> ⚠️ Todos los datos son **sintéticos** (inventados por código). **No representan pacientes reales.**

**Objetivo de aprendizaje.** Un modelo clínico no es más que **una prueba diagnóstica más**, y se juzga con reglas que ya conoces. En este cuaderno entrenamos una **regresión logística** sencilla sobre `pacientes.csv` —no para lucirla, sino para **evaluarla a fondo**— y aprendemos a leer, en lenguaje clínico:

1. La **matriz de confusión** (TP / FP / FN / TN).
2. **Sensibilidad, especificidad, VPP y VPN**, calculadas *a mano* desde esa matriz.
3. El **umbral de decisión**: mover el corte y ver cómo cambia todo.
4. Las **curvas ROC/AUC** y **precisión-recall** (con su línea base = prevalencia).
5. La **calibración**: que un "20 %" signifique de verdad un 20 %.
6. Un vistazo a las métricas de **regresión** (MAE, R²) sobre el riesgo a 10 años.

Y, sobre todo, entender por qué **la exactitud engaña** cuando la clase que importa es rara.

[Abrir en Colab](PENDIENTE_DRIVE)


## 1. Preparamos los datos (se hace solo)

La primera celda de código **genera los datos sintéticos** del curso si no están ya en la sesión. No hay que descargar nada. Ejecútala y espera al mensaje de confirmación.


In [ ]:
# === Generación de los datos sintéticos del curso (se ejecuta sola) ===
# TODOS LOS DATOS SON SINTÉTICOS. No representan pacientes reales. Semilla fija -> reproducible.
import os, numpy as np, pandas as pd

def generar_datos_clinicos(carpeta="."):
    """Crea los CSV del curso si no están ya en la carpeta. Devuelve la ruta."""
    if os.path.exists(os.path.join(carpeta, "pacientes.csv")):
        return carpeta
    RNG = np.random.default_rng(2026)   # misma semilla en todo el curso

    # --- Centros ---
    AREAS = ["Valencia","Alicante","Castellón","Madrid","Barcelona","Sevilla","Bilbao","Zaragoza"]
    nc = 24
    tipo = RNG.choice(["hospital","centro de salud"], nc, p=[0.35,0.65])
    camas = np.where(tipo=="hospital", RNG.integers(120,900,nc), RNG.integers(0,25,nc))
    nserv = np.where(tipo=="hospital", RNG.integers(12,40,nc), RNG.integers(3,10,nc))
    centros = pd.DataFrame({"centro_id":[f"C{i:03d}" for i in range(1,nc+1)],"tipo":tipo,
        "area":RNG.choice(AREAS,nc),"camas":camas,"n_servicios":nserv,
        "urgencias_dia_media":np.where(tipo=="hospital",RNG.integers(80,320,nc),RNG.integers(5,40,nc)),
        "ratio_mayores65":RNG.uniform(0.12,0.34,nc).round(3)})
    centros.to_csv(os.path.join(carpeta,"centros.csv"), index=False)

    # --- Pacientes (tabla central) ---
    N = 20000
    sexo = RNG.choice(["M","F"], N, p=[0.49,0.51])
    edad = np.clip(np.where(RNG.random(N)<0.6, RNG.normal(58,15,N), RNG.normal(40,12,N)),18,95).round().astype(int)
    p_act = np.clip(0.28-0.0016*(edad-18),0.06,0.28); p_ex = np.clip(0.10+0.004*(edad-18),0.10,0.40); p_nu = 1-p_act-p_ex
    tabaquismo = np.array([RNG.choice(["nunca","ex","activo"],p=[a,b,c]) for a,b,c in zip(p_nu,p_ex,p_act)])
    actividad = RNG.choice(["baja","media","alta"], N, p=[0.4,0.4,0.2])
    antec = RNG.binomial(1,0.22,N)
    actn = pd.Series(actividad).map({"baja":1.5,"media":0.0,"alta":-1.3}).values
    imc = np.clip(RNG.normal(26.5,4.2,N)+0.03*(edad-50)+actn+RNG.normal(0,1.0,N),15,48).round(1)
    fa = (tabaquismo=="activo").astype(float); fe = (tabaquismo=="ex").astype(float)
    ta_s = np.clip(RNG.normal(120,12,N)+0.45*(edad-45)+0.7*(imc-25)+6*fa+RNG.normal(0,6,N),85,220).round().astype(int)
    ta_d = np.clip(0.55*ta_s+RNG.normal(20,6,N),50,130).round().astype(int)
    glu = np.clip(RNG.normal(100,12,N)+1.0*(imc-25)+0.3*(edad-45)+9*antec+RNG.normal(0,7,N),60,320).round(1)
    diabetes = (glu>126).astype(int)
    col = np.clip(RNG.normal(195,30,N)+0.4*(edad-45)+1.1*(imc-25)+RNG.normal(0,15,N),110,380).round(1)
    hdl = np.clip(RNG.normal(55,12,N)-0.25*(imc-25)+6*(sexo=="F")-5*fa-3*actn+RNG.normal(0,6,N),20,110).round(1)
    z = (-3.1+0.062*(edad-50)+0.019*(ta_s-120)+0.008*(col-190)-0.028*(hdl-55)+0.055*(imc-25)
         +0.85*fa+0.30*fe+0.60*diabetes+0.45*antec+0.55*(sexo=="M")
         +0.012*np.maximum(edad-65,0)*fa+RNG.normal(0,0.35,N))
    p = 1/(1+np.exp(-z))
    riesgo = (100*p).round(1); evento = RNG.binomial(1,p)
    pacientes = pd.DataFrame({"paciente_id":[f"P{i:05d}" for i in range(1,N+1)],"edad":edad,"sexo":sexo,
        "imc":imc,"ta_sistolica":ta_s,"ta_diastolica":ta_d,"glucemia":glu,"colesterol_total":col,"hdl":hdl,
        "tabaquismo":tabaquismo,"actividad_fisica":actividad,"antecedentes_familiares":antec,
        "diabetes":diabetes,"riesgo_cv_10a":riesgo,"evento_cv":evento})
    pacientes.to_csv(os.path.join(carpeta,"pacientes.csv"), index=False)

    # --- Versión "sucia" para EDA ---
    d = pacientes.copy()
    d["sexo"] = d["sexo"].map(lambda s: RNG.choice({"M":["M","m","Masculino","H","Hombre"],"F":["F","f","Femenino","Mujer"]}[s]))
    idx = RNG.choice(d.index,int(N*0.06),replace=False)
    d["glucemia"] = d["glucemia"].astype(object)   # permite mezclar texto (mmol/L con coma) y números
    d.loc[idx,"glucemia"] = [str(round(v/18.0,1)).replace(".",",") for v in d.loc[idx,"glucemia"]]
    prob_na = np.where(pacientes["edad"]<40,0.12,0.04); mask = RNG.random(N)<prob_na
    d.loc[mask,"hdl"] = np.nan
    d.loc[RNG.choice(d.index,int(N*0.05),replace=False),"colesterol_total"] = np.nan
    d.loc[RNG.choice(d.index,25,replace=False),"edad"] = RNG.integers(150,260,25)
    d.loc[RNG.choice(d.index,20,replace=False),"ta_sistolica"] = -RNG.integers(1,90,20)
    d.loc[RNG.choice(d.index,15,replace=False),"imc"] = RNG.uniform(80,200,15).round(1)
    d["colesterol_total"] = d["colesterol_total"].astype(object); d["ta_diastolica"] = d["ta_diastolica"].astype(object)
    d.loc[RNG.choice(d.index,30,replace=False),"colesterol_total"] = "desconocido"
    d.loc[RNG.choice(d.index,20,replace=False),"ta_diastolica"] = "N/D"
    d["tabaquismo"] = d["tabaquismo"].map(lambda s: RNG.choice({"nunca":["nunca","No fumador","NUNCA"],
        "ex":["ex","exfumador","Ex-fumador"],"activo":["activo","Fumador","SI"]}[s]))
    d = pd.concat([d, d.sample(400,random_state=3)], ignore_index=True)
    idx = RNG.choice(d.index,200,replace=False); d.loc[idx,"paciente_id"] = d.loc[idx,"paciente_id"].map(lambda s:" "+s+" ")
    d = d.sample(frac=1,random_state=11).reset_index(drop=True)
    d.to_csv(os.path.join(carpeta,"pacientes_sucio.csv"), index=False)

    # --- Urgencias (serie temporal) ---
    ND = 730; fechas = pd.date_range("2024-01-01", periods=ND, freq="D")
    doy = fechas.dayofyear.values; dow = fechas.dayofweek.values
    fest_set = {(1,1),(1,6),(5,1),(8,15),(10,12),(11,1),(12,6),(12,8),(12,25)}
    festivo = np.array([1 if (f.month,f.day) in fest_set else 0 for f in fechas])
    gripe = np.array([1 if f.month in (12,1,2) else 0 for f in fechas])
    temp = (15+10*np.sin(2*np.pi*(doy-110)/365)+RNG.normal(0,2.5,ND)).round(1)
    est = 1+0.14*np.sin(2*np.pi*(doy-20)/365)
    sem = np.where(dow==0,1.18,np.where(dow>=5,1.10,1.0))
    mu = 120.0*est*sem*(1+0.22*gripe)*(1+0.15*festivo)
    ingresos = RNG.poisson(np.maximum(mu,1)).astype(int)
    pd.DataFrame({"fecha":fechas.strftime("%Y-%m-%d"),"ingresos":ingresos,"festivo":festivo,
        "temporada_gripe":gripe,"temperatura":temp}).to_csv(os.path.join(carpeta,"urgencias_diarias.csv"), index=False)

    # --- Notas clínicas (texto) ---
    plant = {"cardiología":["dolor torácico opresivo de {t} de evolución, irradiado a brazo izquierdo",
        "disnea de esfuerzo progresiva y palpitaciones, antecedente de hipertensión",
        "control de anticoagulación, fibrilación auricular conocida, sin dolor actual",
        "edemas en miembros inferiores y ortopnea, sospecha de insuficiencia cardiaca"],
      "respiratorio":["tos productiva de {t}, fiebre y disnea, auscultación con crepitantes",
        "sibilancias y opresión torácica, antecedente de asma, mala respuesta a inhalador",
        "disnea súbita y dolor pleurítico, saturación disminuida",
        "tos seca persistente y febrícula, contacto con caso respiratorio"],
      "digestivo":["dolor abdominal en fosa iliaca derecha de {t}, náuseas y febrícula",
        "epigastralgia y pirosis, relación con las comidas, sin signos de alarma",
        "diarrea de {t} sin productos patológicos, buen estado general",
        "ictericia y coluria, dolor en hipocondrio derecho"],
      "neurología":["cefalea intensa de inicio súbito, la peor de su vida, con fotofobia",
        "focalidad neurológica de {t}, debilidad en hemicuerpo y disartria",
        "mareo con giro de objetos y vómitos, sin cortejo vegetativo",
        "crisis comicial presenciada, recuperación progresiva del nivel de conciencia"],
      "traumatología":["caída casual con dolor e impotencia funcional en muñeca, deformidad",
        "lumbalgia mecánica de {t} tras esfuerzo, sin déficit neurológico",
        "esguince de tobillo con edema y dificultad para la deambulación",
        "gonalgia y derrame articular tras traumatismo deportivo"]}
    prio = {"cardiología":[0.35,0.45,0.20],"respiratorio":[0.30,0.45,0.25],"digestivo":[0.20,0.50,0.30],
        "neurología":[0.45,0.35,0.20],"traumatología":[0.12,0.48,0.40]}
    tiempos = ["horas","un día","dos días","una semana","varios días"]; esp_list = list(plant.keys()); rows=[]
    for _ in range(3000):
        e = RNG.choice(esp_list); txt = RNG.choice(plant[e]).replace("{t}",RNG.choice(tiempos))
        rows.append((txt,e,RNG.choice(["alta","media","baja"],p=prio[e]),RNG.choice(centros["centro_id"].values)))
    pd.DataFrame(rows,columns=["texto","especialidad","prioridad","centro_id"]).to_csv(os.path.join(carpeta,"notas_clinicas.csv"),index=False)

    # --- Wearable (señal) ---
    rows=[]
    for s in range(1,201):
        fcb=RNG.normal(66,6); pb=RNG.normal(7500,2500); sb=RNG.normal(7.0,0.8); anom=RNG.random()<0.05
        for dia in range(1,31):
            fc=fcb+RNG.normal(0,3); pasos=max(0,pb+RNG.normal(0,1500)); sue=float(np.clip(sb+RNG.normal(0,0.6),3,11))
            if anom and dia in (14,15,16): fc+=RNG.uniform(18,30); sue-=RNG.uniform(1.5,3)
            rows.append((f"S{s:03d}",dia,round(fc,1),int(pasos),round(sue,1)))
    pd.DataFrame(rows,columns=["sujeto_id","dia","fc_reposo","pasos","horas_sueno"]).to_csv(os.path.join(carpeta,"wearable.csv"),index=False)
    return carpeta

generar_datos_clinicos(".")
print("Datos sintéticos listos en la carpeta actual. (Recuerda: son datos SINTÉTICOS, no reales.)")

## 2. Conoce tus datos antes de evaluar nada

Abrimos la tabla de pacientes y miramos las **primeras filas**. Cada **fila** es un paciente; cada **columna**, un dato suyo. Nos fijaremos en dos columnas objetivo: `riesgo_cv_10a` (riesgo cardiovascular a 10 años, en %) y `evento_cv` (0 = no sufrió evento, 1 = sí).


In [ ]:
import pandas as pd

pacientes = pd.read_csv("pacientes.csv")
pacientes.head()

**Lo que vemos:** una cohorte con variables clínicas habituales (edad, IMC, tensión, glucemia, colesterol, HDL, hábitos…) y, al final, las dos columnas que queremos predecir. Aquí trabajaremos la **clasificación** de `evento_cv` (sí/no) y, al final, un vistazo a la **regresión** de `riesgo_cv_10a` (un porcentaje).


### La cifra más importante del cuaderno: la prevalencia

Antes de evaluar nada, hay que saber **cómo de frecuente es lo que buscamos**. La proporción de pacientes con `evento_cv = 1` se llama **prevalencia**. La calculamos y la dibujamos.


In [ ]:
prevalencia = pacientes["evento_cv"].mean()
print(f"Pacientes totales: {len(pacientes):,}")
print(f"Con evento (evento_cv = 1): {pacientes['evento_cv'].sum():,}")
print(f"Prevalencia: {prevalencia:.1%}")

In [ ]:
import matplotlib.pyplot as plt

conteo = pacientes["evento_cv"].value_counts().sort_index()
plt.figure(figsize=(6, 4))
plt.bar(["Sin evento (0)", "Con evento (1)"], conteo.values,
        color=["#9fb3c8", "#e8663a"], edgecolor="white")
plt.ylabel("Nº de pacientes")
plt.title("Reparto de la clase objetivo (datos sintéticos)")
for i, v in enumerate(conteo.values):
    plt.text(i, v, f"{v:,}", ha="center", va="bottom")
plt.show()

**Lo que vemos aquí es** un problema **desbalanceado**: alrededor de **1 de cada 5** pacientes sufre el evento (prevalencia ≈ 19 %); los otros 4 de cada 5, no. Esta asimetría es la que hace **peligrosa** a la métrica más intuitiva de todas, la exactitud, como comprobaremos enseguida. Grábate el número: **la clase que nos importa es la minoritaria.**


## 3. Entrenamos una regresión logística (para tener probabilidades)

No buscamos el mejor modelo del mundo —eso es de otra unidad—, sino uno **sencillo y honesto** que nos dé, para cada paciente, una **probabilidad** de evento sobre la que aprender a evaluar.

Primero separamos los datos en **entrenamiento** (el modelo aprende aquí) y **test** (lo reservamos para juzgarlo con honestidad). Usamos `stratify` para que la prevalencia (≈ 19 %) sea la misma en ambos lados, y escalamos/codificamos **solo con el entrenamiento** (dentro de un *pipeline*) para **no filtrar información** del test (evitar la *fuga de datos*).


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

# Variables de entrada: numéricas y categóricas
num = ["edad", "imc", "ta_sistolica", "ta_diastolica", "glucemia",
       "colesterol_total", "hdl", "antecedentes_familiares", "diabetes"]
cat = ["sexo", "tabaquismo", "actividad_fisica"]

X = pacientes[num + cat]
y = pacientes["evento_cv"]

# El preprocesado va DENTRO del pipeline -> se ajusta solo con el train (sin fuga)
pre = ColumnTransformer([
    ("num", StandardScaler(), num),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat),
])

Xtr, Xte, ytr, yte = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y)

print("Pacientes para entrenar:", len(Xtr))
print("Pacientes reservados para evaluar (test):", len(Xte))
print(f"Prevalencia en train: {ytr.mean():.1%}   |   en test: {yte.mean():.1%}")

**Lo que hemos hecho:** repartir los pacientes en dos grupos que **no se solapan**, con la misma proporción de eventos en cada uno. Todo lo que midamos a partir de ahora será sobre el **test**, pacientes que el modelo **no ha visto** al aprender. Esa es la única forma honesta de estimar cómo se comportará con pacientes nuevos.


Ahora **entrenamos** la regresión logística y le pedimos, para cada paciente del test, su **probabilidad** de evento (un número entre 0 y 1). El modelo **no dice "sí/no"**: da una *nota de sospecha*.


In [ ]:
modelo = Pipeline([("pre", pre), ("clf", LogisticRegression(max_iter=2000))])
modelo.fit(Xtr, ytr)

# Probabilidad de evento (columna 1 = clase positiva) para cada paciente del test
prob = modelo.predict_proba(Xte)[:, 1]

print("Modelo entrenado. Probabilidades de los 8 primeros pacientes del test:")
print(prob[:8].round(3))

**Lo que vemos:** para cada paciente, un número entre 0 y 1. *"A este le veo un 0,34 de tener un evento."* Cuanta más nota, más sospecha. Toda la evaluación que sigue consiste en **decidir qué hacemos con esas notas** y en **comprobar si son de fiar**.


## 4. Por qué la exactitud engaña (la trampa del desbalanceo)

La **exactitud** (*accuracy*) es la proporción de aciertos sobre el total. Suena razonable... y es traicionera. Para verlo, comparamos dos "modelos":

* Un **modelo perezoso** que dice **"nadie tendrá evento"** a todo el mundo.
* Nuestra **regresión logística** decidiendo con el umbral por defecto (0,5).

Fíjate en la exactitud de ambos **y**, sobre todo, en cuántos pacientes en riesgo detecta cada uno.


In [ ]:
import numpy as np

# --- Modelo perezoso: predice SIEMPRE 0 (ningún evento) ---
pred_perezoso = np.zeros(len(yte), dtype=int)
acc_perezoso = (pred_perezoso == yte.values).mean()
detectados_perezoso = ((pred_perezoso == 1) & (yte.values == 1)).sum()

# --- Regresión logística con umbral 0,5 ---
pred_modelo = (prob >= 0.5).astype(int)
acc_modelo = (pred_modelo == yte.values).mean()
detectados_modelo = ((pred_modelo == 1) & (yte.values == 1)).sum()
enfermos_reales = int(yte.sum())

print(f"Enfermos reales en el test: {enfermos_reales}")
print()
print(f"Modelo PEREZOSO ('nadie tendrá evento'):")
print(f"   exactitud = {acc_perezoso:.1%}   |   enfermos detectados = {detectados_perezoso} de {enfermos_reales}")
print()
print(f"Regresión LOGÍSTICA (umbral 0,5):")
print(f"   exactitud = {acc_modelo:.1%}   |   enfermos detectados = {detectados_modelo} de {enfermos_reales}")

**Lo que vemos —y es demoledor:** el modelo **perezoso**, que no detecta **ni un solo** paciente en riesgo, saca una exactitud de ~81 % solo porque acierta a todos los sanos (que son mayoría). Su sensibilidad es **0 %**: clínicamente **inútil**, pero "aprueba" con nota alta si miramos la exactitud.

Nuestra regresión logística tiene una exactitud **parecida**, pero sí encuentra pacientes reales. La lección: **con clases desbalanceadas, la exactitud a solas no distingue lo útil de lo inútil.** Necesitamos métricas que miren la clase minoritaria por separado — y para eso vamos a la matriz de confusión.


## 5. La matriz de confusión (TP / FP / FN / TN)

La **matriz de confusión** cruza lo que el modelo predijo con lo que era verdad. En un problema binario tiene cuatro celdas. Tomando como "positivo" que el modelo prediga **en riesgo** (`evento_cv = 1`):

* **TP** (verdadero positivo): dice *en riesgo* y el paciente **sí** tuvo el evento → **acierto útil**.
* **TN** (verdadero negativo): dice *no en riesgo* y **no** lo tuvo → **acierto rutinario**.
* **FP** (falso positivo): dice *en riesgo* a quien **no** lo tendrá → **falsa alarma** (pruebas y ansiedad de más).
* **FN** (falso negativo): dice *no en riesgo* a quien **sí** lo tendrá → **fallo silencioso** (paciente sin su prevención).

La calculamos (con el umbral 0,5) y la **dibujamos** en la orientación clínica clásica, la tabla 2×2 de *prueba × enfermedad*.


In [ ]:
from sklearn.metrics import confusion_matrix

# confusion_matrix devuelve, en orden: TN, FP, FN, TP
tn, fp, fn, tp = confusion_matrix(yte, pred_modelo).ravel()
print(f"TP (aciertos 'en riesgo')      = {tp}")
print(f"TN (aciertos 'no en riesgo')   = {tn}")
print(f"FP (falsas alarmas)            = {fp}")
print(f"FN (fallos silenciosos)        = {fn}")

In [ ]:
# Dibujamos la matriz en orientación clínica: filas = decisión del modelo, columnas = verdad
M = np.array([[tp, fp],
              [fn, tn]])
etiquetas = np.array([["TP\nacierto útil", "FP\nfalsa alarma"],
                      ["FN\nfallo silencioso", "TN\nacierto rutinario"]])
# Verde = acierto (diagonal TP, TN) ; rojo = error (FP, FN)
color_fondo = np.array([[1.0, 0.0],
                        [0.0, 1.0]])

fig, ax = plt.subplots(figsize=(7, 5))
ax.imshow(color_fondo, cmap="RdYlGn", vmin=-0.35, vmax=1.35, alpha=0.45)
for i in range(2):
    for j in range(2):
        ax.text(j, i, f"{etiquetas[i, j]}\n\n{M[i, j]}",
                ha="center", va="center", fontsize=12)
ax.set_xticks([0, 1]); ax.set_xticklabels(["Evento real: Sí", "Evento real: No"])
ax.set_yticks([0, 1]); ax.set_yticklabels(["Modelo:\nen riesgo (+)", "Modelo:\nno en riesgo (−)"])
ax.set_title("Matriz de confusión — regresión logística (umbral 0,5)")
plt.tight_layout()
plt.show()

**Lo que vemos aquí es** el resumen completo del comportamiento del modelo en cuatro números. Los aciertos están en la **diagonal verde** (TP arriba-izquierda, TN abajo-derecha); los dos tipos de error, en **rojo**. Fíjate especialmente en los **FN** (abajo-izquierda): son los pacientes en riesgo que el modelo **dejó pasar**. En prevención cardiovascular, ese es el error que más suele doler. Todas las métricas clínicas que vienen ahora **nacen de estas cuatro celdas**.


## 6. Sensibilidad, especificidad, VPP y VPN — a mano

No hace falta ninguna librería mágica: las cuatro métricas clínicas salen de **dividir** las celdas de la matriz. Las calculamos nosotros mismos para que se vea de dónde vienen.

| Métrica | Fórmula | Pregunta que responde |
| --- | --- | --- |
| **Sensibilidad** (= recall) | TP / (TP + FN) | De los enfermos, ¿a cuántos detecto? |
| **Especificidad** | TN / (TN + FP) | De los sanos, ¿a cuántos descarto bien? |
| **VPP** (= precisión) | TP / (TP + FP) | Si digo "en riesgo", ¿acierto? |
| **VPN** | TN / (TN + FN) | Si digo "no en riesgo", ¿acierto? |


In [ ]:
sensibilidad = tp / (tp + fn)
especificidad = tn / (tn + fp)
vpp = tp / (tp + fp)
vpn = tn / (tn + fn)

print(f"Sensibilidad (recall)   = {sensibilidad:.1%}   -> de {tp+fn} enfermos, detecta {tp}")
print(f"Especificidad           = {especificidad:.1%}   -> de {tn+fp} sanos, descarta bien {tn}")
print(f"VPP (precisión)         = {vpp:.1%}   -> de {tp+fp} 'en riesgo', acierta {tp}")
print(f"VPN                     = {vpn:.1%}   -> de {tn+fn} 'no en riesgo', acierta {tn}")

**Cómo leer estos números en la consulta:**

* La **sensibilidad** dice qué porcentaje de los pacientes que *sí* tendrán el evento logramos captar. Si es baja, hay muchos **fallos silenciosos (FN)**: gente en riesgo que pasa desapercibida.
* La **especificidad** dice cuántos sanos descartamos correctamente. Si es baja, hay muchas **falsas alarmas (FP)**.
* El **VPP** responde a lo que más importa cuando el modelo marca a alguien: *"si me dice en riesgo, ¿cuánto me lo creo?"*.
* El **VPN** es la fiabilidad del *"puede irse tranquilo"*.

Con el umbral 0,5, este modelo es **específico pero poco sensible**: descarta bien a los sanos, pero se le escapan bastantes pacientes en riesgo. Eso, en un cribado, no nos vale. La buena noticia: **el umbral lo elegimos nosotros.**


## 7. El umbral de decisión es una decisión clínica

El 0,5 no tiene nada de sagrado: es solo el corte por defecto. **Podemos moverlo** para alinearlo con lo que nos importa:

* **Umbral bajo** (p. ej. 0,15) → modo *cribado*: capturamos a casi todos los que tendrán evento (**sensibilidad alta**), a costa de más **falsas alarmas** (especificidad y VPP bajan).
* **Umbral alto** (p. ej. 0,80) → modo *confirmación*: casi no damos falsas alarmas (**especificidad y VPP altos**), pero se nos **escapan** pacientes en riesgo.

Lo comprobamos en una **tabla**: para varios umbrales, recalculamos sensibilidad, especificidad y VPP.


In [ ]:
def metricas_en_umbral(prob, y_real, umbral):
    pred = (prob >= umbral).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_real, pred).ravel()
    sens = tp / (tp + fn) if (tp + fn) else 0
    esp = tn / (tn + fp) if (tn + fp) else 0
    vpp = tp / (tp + fp) if (tp + fp) else 0
    marcados = pred.mean()   # % de pacientes marcados como 'en riesgo'
    return {"Umbral": umbral, "Sensibilidad": sens, "Especificidad": esp,
            "VPP": vpp, "% marcados en riesgo": marcados}

tabla = pd.DataFrame([metricas_en_umbral(prob, yte, u)
                      for u in [0.10, 0.15, 0.30, 0.50, 0.70, 0.85]])
# Formateamos como porcentajes para leerla como en la clínica
tabla_fmt = tabla.copy()
for col in ["Sensibilidad", "Especificidad", "VPP", "% marcados en riesgo"]:
    tabla_fmt[col] = (tabla_fmt[col] * 100).round(1).astype(str) + " %"
tabla_fmt

**Lo que vemos aquí es** el *dial* entre dos males, número a número. Al **bajar** el umbral (arriba de la tabla), la **sensibilidad sube** —cazamos más enfermos— pero la **especificidad baja** y marcamos a mucha más gente como "en riesgo". Al **subir** el umbral (abajo), pasa lo contrario. No hay un corte mágico que mejore ambas cosas: **lo que ganas por un lado, lo pagas por el otro.**

Elegir dónde poner la raya es, por tanto, una **decisión clínica**: depende de cuál de los dos errores cuesta más. Si un **falso negativo** (dejar sin prevención a quien la necesita) es lo más grave, **bajamos** el umbral y aceptamos más falsas alarmas. Ningún algoritmo puede tomar esa decisión por ti.


## 8. La curva ROC y el AUC (todos los umbrales a la vez)

Como el umbral lo eliges tú, tiene sentido ver el modelo **para todos los umbrales posibles de golpe**. La **curva ROC** traza la **sensibilidad** (eje vertical, la quieres alta) frente a la **tasa de falsos positivos** = 1 − especificidad (eje horizontal, la quieres baja). Cada punto de la curva es **un umbral**. Cuanto más pegada a la **esquina superior izquierda**, mejor.


In [ ]:
from sklearn.metrics import roc_curve, roc_auc_score

fpr, tpr, _ = roc_curve(yte, prob)
auc = roc_auc_score(yte, prob)

plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, color="#00AEC7", lw=2, label=f"Regresión logística (AUC = {auc:.2f})")
plt.plot([0, 1], [0, 1], "--", color="gray", label="Azar (AUC = 0,50)")
plt.xlabel("1 − especificidad  (tasa de falsos positivos)")
plt.ylabel("Sensibilidad  (verdaderos positivos)")
plt.title("Curva ROC")
plt.legend(loc="lower right")
plt.show()
print(f"AUC = {auc:.3f}")

**Lo que vemos aquí es** una curva que se despega claramente de la diagonal (el azar) y se acerca a la esquina buena. El **AUC ≈ 0,84** resume esa curva en un número: es la probabilidad de que, tomando al azar un paciente **con** evento y uno **sin** evento, el modelo le asigne **más riesgo al que sí lo tuvo**. Es decir, mide si el modelo sabe **ordenar** a los pacientes, sin comprometerse aún con un umbral.

Un AUC de 0,84 es una capacidad de ordenación **notable** para un modelo tan simple. Pero cuidado: ordenar bien **no** garantiza que las probabilidades sean creíbles (lo veremos en la calibración), ni cuenta bien la historia cuando la clase positiva es rara — para eso está la curva que viene ahora.


## 9. La curva precisión-recall (mejor cuando el positivo es raro)

La ROC tiene un punto ciego: cuando hay **muchísimos** negativos fáciles (como aquí, con prevalencia ≈ 19 %), esos verdaderos negativos "inflan" la parte buena de la curva y dan una impresión **demasiado optimista**.

La **curva precisión-recall (PR)** evita ese problema porque **solo mira la clase positiva**: enfrenta el **VPP (precisión)** al **recall (sensibilidad)** a lo largo de todos los umbrales. Su **línea base** (lo que haría decidir al azar) **no** es 0,5, sino la **prevalencia** (≈ 0,19).


In [ ]:
from sklearn.metrics import precision_recall_curve, average_precision_score

precision, recall, _ = precision_recall_curve(yte, prob)
ap = average_precision_score(yte, prob)
prevalencia_test = yte.mean()

plt.figure(figsize=(6, 5))
plt.plot(recall, precision, color="#7b3fa0", lw=2, label=f"Regresión logística (PR-AUC = {ap:.2f})")
plt.axhline(prevalencia_test, ls="--", color="gray",
            label=f"Línea base = prevalencia ({prevalencia_test:.2f})")
plt.xlabel("Recall = sensibilidad  (de los enfermos, cuántos cazo)")
plt.ylabel("Precisión = VPP  (de mis alarmas, cuántas aciertan)")
plt.title("Curva precisión-recall")
plt.legend(loc="upper right")
plt.show()
print(f"PR-AUC (precisión media) = {ap:.3f}   |   línea base (azar) = {prevalencia_test:.3f}")

**Lo que vemos aquí es** cómo se degrada la fiabilidad de las alarmas (**precisión / VPP**, eje vertical) a medida que exigimos cazar más enfermos (**recall**, eje horizontal). La clave está en la **línea base**: un modelo que decidiera al azar quedaría en ≈ 0,19; nuestra curva vive **muy por encima**, así que aporta valor real sobre la clase que importa.

**Cuándo usar cada curva:** la **ROC/AUC** es cómoda para **comparar modelos** rápido; la **PR** es más **honesta cuando la clase positiva es rara** (cribados, eventos), porque no se deja engañar por la abundancia de sanos. Con esta prevalencia, conviene mirar **las dos**.


## 10. Calibración: que un 20 % signifique de verdad un 20 %

Un modelo puede **ordenar** de maravilla (AUC alto) y aun así **mentir en los números**: decir "20 % de riesgo" cuando la frecuencia real en ese grupo es del 40 %. Ordenar bien y estar bien calibrado son cosas **distintas**, y en clínica la segunda suele importar **más**, porque las decisiones (¿ofrecer estatinas?, ¿qué le digo al paciente?) usan **la probabilidad en sí**, no solo el orden.

La **curva de calibración** compara, por tramos de probabilidad, lo que el modelo **predijo** (eje horizontal) con lo que **realmente pasó** (eje vertical). El ideal es la **diagonal**.


In [ ]:
from sklearn.calibration import calibration_curve

frac_real, prob_media = calibration_curve(yte, prob, n_bins=10, strategy="quantile")

plt.figure(figsize=(6, 5))
plt.plot([0, 1], [0, 1], "--", color="gray", label="Calibración perfecta")
plt.plot(prob_media, frac_real, "o-", color="#e8663a", label="Regresión logística")
plt.xlabel("Probabilidad predicha por el modelo")
plt.ylabel("Frecuencia real de eventos observada")
plt.title("Curva de calibración (reliability diagram)")
plt.legend(loc="upper left")
plt.show()

**Lo que vemos aquí es** que los puntos del modelo caen **muy cerca de la diagonal**: cuando dice "30 % de riesgo", de ese grupo acaba teniendo el evento cerca de un 30 %. Es decir, sus probabilidades son **de fiar**. (La regresión logística suele salir bien calibrada *de fábrica*; otros modelos, como ciertos *ensembles* de árboles, no, y hay que recalibrarlos.)

**Por qué importa clínicamente:** si el modelo estuviera mal calibrado —dijera 20 % cuando es 40 %— llevaría a **infratratar** a pacientes que sí lo necesitaban, por muy bueno que fuera su "ranking". Regla de oro: **reporta siempre la calibración junto al AUC, no en su lugar.**


## 11. Un vistazo a la regresión: MAE y R² en puntos de riesgo

Hasta aquí hemos evaluado una **clasificación** (evento sí/no). Pero `riesgo_cv_10a` es un **número** (un porcentaje), y predecir números es **regresión**. Sus métricas se leen distinto:

* **MAE** (error absoluto medio): de media, ¿por cuántos **puntos de riesgo** nos desviamos? Está en las mismas unidades que el riesgo, así que es muy fácil de comunicar.
* **R²**: ¿cuánto mejor es el modelo que predecir a todos el **riesgo medio** de la cohorte? (1 = perfecto; 0 = no aporta nada.)

Entrenamos una **regresión lineal** sencilla sobre el riesgo y medimos ambas en el test.


In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score

y_riesgo = pacientes["riesgo_cv_10a"]
Xtr_r, Xte_r, ytr_r, yte_r = train_test_split(
    X, y_riesgo, test_size=0.25, random_state=42)

reg = Pipeline([("pre", pre), ("clf", LinearRegression())])
reg.fit(Xtr_r, ytr_r)
pred_riesgo = reg.predict(Xte_r)

mae = mean_absolute_error(yte_r, pred_riesgo)
r2 = r2_score(yte_r, pred_riesgo)
print(f"MAE = {mae:.2f} puntos de riesgo   (de media nos desviamos ~{mae:.1f} puntos)")
print(f"R²  = {r2:.3f}")

In [ ]:
# Predicho vs real: cuanto más cerca de la diagonal, mejor
plt.figure(figsize=(5.5, 5.5))
plt.scatter(yte_r, pred_riesgo, s=6, alpha=0.25, color="#00AEC7")
lims = [0, max(yte_r.max(), pred_riesgo.max())]
plt.plot(lims, lims, "--", color="gray", label="Predicción perfecta")
plt.xlabel("Riesgo real a 10 años (%)")
plt.ylabel("Riesgo predicho por el modelo (%)")
plt.title("Regresión del riesgo: predicho vs real")
plt.legend(loc="upper left")
plt.show()

**Lo que vemos aquí es** una nube de puntos alineada con la diagonal: el modelo sigue bien la tendencia del riesgo real. Un **MAE de ~7 puntos** significa que, de media, nos equivocamos en unos 7 puntos porcentuales de riesgo —fácil de explicar a un comité—, y un **R² ≈ 0,8** dice que explicamos la mayor parte de la variabilidad del riesgo entre pacientes (mucho mejor que predecir la media a todos).

Un matiz importante: un MAE de 7 no es "bueno" o "malo" en abstracto; solo cobra sentido **comparado con un baseline** (predecir siempre el riesgo medio) y, mejor aún, con una **escala de riesgo clínica** ya validada. Sin ese contraste, un número suelto no dice gran cosa.


## 12. Qué hemos aprendido

* **La exactitud engaña con clases desbalanceadas.** Con prevalencia ≈ 19 %, un modelo que no detecta a nadie saca ~81 % de exactitud y es **clínicamente inútil**. Mira **sensibilidad, especificidad y valores predictivos**, nunca la exactitud a solas.
* **Todo nace de la matriz de confusión** (TP, FP, FN, TN). Sensibilidad, especificidad, VPP y VPN son simples divisiones de esas cuatro celdas.
* **El umbral es una decisión clínica**, no el 0,5 por defecto: bajarlo prioriza la sensibilidad (modo cribado); subirlo, la especificidad y el VPP (modo confirmación). Decide **qué error duele más** y elige en consecuencia.
* **ROC/AUC** mide si el modelo **ordena** bien; la **curva PR** (con línea base = prevalencia) es más honesta cuando el positivo es **raro**.
* **Calibración ≠ ordenación:** que un 20 % signifique de verdad un 20 %. Repórtala **junto** al AUC.
* En **regresión**, lee el **MAE** en puntos de riesgo y el **R²** frente a predecir la media — siempre contra un **baseline**.

### 🤖 Cómo pedírselo a un asistente de IA (conexión con la U10)

No tienes que memorizar este código. Puedes describir lo que quieres y dejar que la IA lo escriba; tu papel es **dar el contexto clínico** y **leer los resultados con criterio**. Un ejemplo de encargo:

```
Con 'pacientes.csv' (target de clasificación: evento_cv, prevalencia ≈ 19 %),
en español y por celdas:
1. Separa train/test estratificado y sin fuga (escala solo con el train).
2. Entrena una regresión logística que dé probabilidades.
3. Muestra la matriz de confusión y calcula a mano sensibilidad, especificidad,
   VPP y VPN. Comenta la trampa de la exactitud con esta prevalencia.
4. Dibuja la curva ROC (con AUC), la precisión-recall (línea base = prevalencia)
   y la curva de calibración.
5. Como NO detectar un evento (falso negativo) es el error más caro, dime a qué
   umbral moverías el corte y recalcula la matriz de confusión con ese umbral.
```

Revisa siempre que la partición sea **honesta**, que la matriz esté **bien orientada**, que se mire la **calibración** (no solo el AUC) y que la métrica destacada corresponda al **coste clínico del error**.
